# AWF LLM Training — GPU Version (Google Colab)

This notebook trains the AWF LLM on a GPU (Google Colab T4 is free).

**What you get**: A 622K-parameter AWF LLM trained on TinyStories that generates coherent text.

**On Colab GPU (T4)**: ~10x faster than CPU. 1 hour of GPU = ~10 hours of CPU training.

## Instructions
1. Set Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Training auto-resumes from checkpoint if you re-run

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/Deexv/AWF.git
%cd AWF
!pip install -r requirements.txt
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
# Download TinyStories dataset (25MB subset)
!python scripts/download_tinystories.py

## Training

The training script auto-detects GPU. On Colab T4, each epoch takes ~5 minutes (vs ~50 min on CPU).

Run the cell below to train for 1 epoch (~5 min on GPU). Re-run to continue training (auto-resumes from checkpoint).

For a well-trained model, run 10-20 epochs (50-100 minutes on GPU).

In [ ]:
# Train AWF model (auto-resumes from checkpoint)
# Adjust --epochs and --time_budget as needed
# On GPU, you can do 10+ epochs in one run
!python scripts/train.py --resume --epochs 5 --time_budget 1800 --lr 5e-4 \
    --datasets data/tinystories_train.txt \
    --batch_size 32

In [ ]:
# Train more if you want better quality (re-run as many times as needed)
# Lower LR for fine-tuning after initial training
!python scripts/train.py --resume --epochs 5 --time_budget 1800 --lr 2e-4 \
    --batch_size 32

## Chat with the trained model

Now you can chat with your AWF LLM!

In [ ]:
# Generate text samples
!python scripts/chat_v2.py --prompt "Once upon a time there was a little" --tokens 200 --temperature 0.7

In [ ]:
# Interactive chat (won't work well in Colab cell — run locally instead)
# For demo, generate from multiple prompts:
!python scripts/chat_v2.py

In [ ]:
# Run the benchmark to compare AWF vs Dense
# First train the dense baseline:
!python scripts/train.py --model dense --epochs 3 --time_budget 1800 --batch_size 32
# Then run benchmark:
!python scripts/benchmark_10m.py

## Save checkpoint to Google Drive (optional)

If you want to keep your trained model, save it to Google Drive.

In [ ]:
# Mount Google Drive and copy checkpoint
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/AWF
!cp checkpoints/awf_10m.pt /content/drive/MyDrive/AWF/
!cp -r benchmarks /content/drive/MyDrive/AWF/
print('Checkpoint saved to Google Drive: /content/drive/MyDrive/AWF/')

## Train on multiple datasets (optional)

The training script supports multiple datasets. They get concatenated automatically.

In [ ]:
# Example: train on TinyStories + your own text file
# Upload your text file to Colab first, then:
# !python scripts/train.py --resume --epochs 2 --datasets data/tinystories_train.txt /content/my_text.txt --batch_size 32